# Multiple Regression — Boston Housing (Keras)

Uses **all 13 features** with a small fully-connected neural network.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

## Load data

In [ ]:
df = pd.read_csv('datasets/Boston1.csv')
df.rename(columns={'medv': 'price'}, inplace=True)
features = [c for c in df.columns if c != 'price']
X = df[features]
y = df['price']
features

## Split + scale

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

## Build & train

In [ ]:
model = Sequential([
    Input(shape=(X_train_s.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1),
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train_s, y_train, epochs=200, batch_size=16,
                    validation_split=0.2, verbose=0)
print('Final val_mae:', history.history['val_mae'][-1])

## Evaluate

In [ ]:
y_pred = model.predict(X_test_s, verbose=0).flatten()
print(f'MAE: {mean_absolute_error(y_test, y_pred):.4f}')
print(f'R²:  {r2_score(y_test, y_pred):.4f}')

## Save

In [ ]:
MODELS_DIR = Path('models'); MODELS_DIR.mkdir(parents=True, exist_ok=True)
model.save(MODELS_DIR / 'boston_multi.keras')
joblib.dump(scaler, MODELS_DIR / 'boston_multi_scaler.pkl')
joblib.dump(features, MODELS_DIR / 'boston_multi_features.pkl')
print('Saved -> models/boston_multi.keras + scaler + feature list')